<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/_kids_high_school_derivatives_and_slopes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualizing the Derivative: A Mobile-First Math Journey

### Overview for Educators & Parents
This notebook generates a high-fidelity mathematical animation designed specifically for mobile-first social media platforms (9:16 aspect ratio). The video guides students through one of the most fundamental shifts in Calculus: moving from an average rate of change (Secant) to an instantaneous rate of change (Tangent).

**Key Concepts Explored:**
*   **The Secant Line:** Visualizing the slope between two distant points P and Q.
*   **The Limit Process:** Watching point Q travel along the curve f(x)=x^2 until it practically merges with P.
*   **The Tangent Line:** Defining the derivative as the precise slope at a single point.
*   **Local Linearity (The 'Zoom' Secret):** A powerful visual proof showing that even the curviest functions look like straight lines when you look close enough. This is a core intuition for understanding why derivatives work.

---

### Project Metadata
*   **Author:** Mugambi Ndwiga
*   **Instagram:** [@MugambiNdwiga_math](https://www.instagram.com/MugambiNdwiga_math)
*   **Repository:** [GitHub: Mathematical video animations and visualization](https://github.com/zombimann/Mathematical-video-animations-and-visualization)

### Setup & Rendering
The following cells install the necessary Manim environment and LaTeX dependencies to render the animation directly in Google Colab.

In [2]:
# 1. Update package list
!sudo apt-get update

# 2. Install Manim system dependencies + LaTeX suite
!sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg \
    texlive texlive-latex-extra texlive-fonts-extra \
    texlive-latex-recommended texlive-science dvisvgm

# 3. Pin NumPy to avoid Manim/Numpy 2.0 compatibility issues
!pip install "manim>=0.18.0" "numpy<2.0.0"

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.3 MB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,043 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,344 kB]
Get:14 http://security.ubu

In [1]:
%load_ext manim

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


The manim module is not an IPython extension.


In [3]:
%%manim -v WARNING --format=mp4 -o derivative_tangent DerivativeAsTangentSlope
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Level: High School
Concept: Derivative as tangent slope — the secant-to-tangent limit, zoom-to-linear insight
Repository: github.com/zombimann/Mathematical-video-animations-and-visualization
"""

from __future__ import annotations
from dataclasses import dataclass
import numpy as np
import sympy as sp
from manim import *

# ─────────────────────────────────────────────
#  Portrait / YouTube-Shorts configuration
# ─────────────────────────────────────────────
config.pixel_width  = 1080
config.pixel_height = 1920
config.frame_width  = 9
config.frame_height = 16
config.background_color = "#1E222A"
config.frame_rate   = 30


# ─────────────────────────────────────────────
#  Design tokens  (single source of truth)
# ─────────────────────────────────────────────
@dataclass(frozen=True)
class Style:
    bg:           str = "#1E222A"
    grid:         str = "#2D3748"
    text:         str = "#F0F0F0"
    muted:        str = "#8899AA"
    cyan:         str = "#00F0FF"
    orange:       str = "#FFB347"
    magenta:      str = "#FF66CC"
    green:        str = "#7EFF8A"
    card_fill:    str = "#252B35"
    card_stroke:  str = "#445063"

S = Style()


# ─────────────────────────────────────────────
#  Math constants  (parametric)
# ─────────────────────────────────────────────
@dataclass(frozen=True)
class Params:
    focus_x:    float = 1.5      # point of interest on the curve
    h_start:    float = 1.2      # initial secant width
    h_end:      float = 0.018    # final secant width (near-zero)
    x_min:      float = -0.5
    x_max:      float = 2.8
    y_min:      float = -0.3
    y_max:      float = 5.5

P = Params()

# Symbolic derivative for correctness display
_x  = sp.Symbol("x")
_fx = _x ** 2
_dfx = sp.diff(_fx, _x)   # = 2x


# ─────────────────────────────────────────────
#  Helper: correct tangent slope in screen space
# ─────────────────────────────────────────────
def _screen_slope(axes: Axes, x: float, dy_dx: float) -> float:
    """Convert data-space slope to screen-space angle, accounting for axis scaling."""
    p0 = axes.c2p(0, 0)
    px = axes.c2p(1, 0)
    py = axes.c2p(0, 1)
    sx = np.linalg.norm(np.array(px) - np.array(p0))   # pixels per x-unit
    sy = np.linalg.norm(np.array(py) - np.array(p0))   # pixels per y-unit
    return dy_dx * (sy / sx)


# ─────────────────────────────────────────────
#  Main scene
# ─────────────────────────────────────────────
class DerivativeAsTangentSlope(MovingCameraScene):

    def construct(self) -> None:
        self.camera.background_color = S.bg

        # ── persistent frame decorations ──
        watermark = self._watermark()
        watermark.add_updater(lambda m: m.to_corner(DL, buff=0.18))
        self.add(watermark)

        # ── build graph objects ──
        axes  = self._make_axes()
        curve = axes.plot(lambda x: x**2,
                          x_range=[P.x_min, P.x_max],
                          color=S.cyan, stroke_width=4)
        curve_label = MathTex(r"f(x) = x^2", color=S.cyan, font_size=34
                               ).next_to(axes.c2p(2.2, 4.84), UR, buff=0.08)

        # ── SCENE 1: hook ────────────────────────
        hook_title = self._title("What is the slope\nof a CURVE?")
        hook_sub   = Text("...at a single point?",
                          color=S.muted, font_size=30,
                          ).next_to(hook_title, DOWN, buff=0.3)

        self.play(FadeIn(hook_title, shift=0.3*DOWN), run_time=0.9)
        self.play(FadeIn(hook_sub,   shift=0.1*DOWN), run_time=0.7)
        self.wait(0.8)
        self.play(Create(axes), run_time=1.2)
        self.play(Create(curve), Write(curve_label), run_time=1.5)
        self.wait(0.5)
        self.play(FadeOut(hook_title), FadeOut(hook_sub), run_time=0.5)

        # ── SCENE 2: mark the focus point ────────────────
        fx_val  = P.focus_x
        fy_val  = fx_val ** 2

        focus_dot = Dot(axes.c2p(fx_val, fy_val),
                        radius=0.10, color=S.orange,
                        z_index=3)

        v_proj = DashedLine(
            axes.c2p(fx_val, 0), axes.c2p(fx_val, fy_val),
            color=S.orange, stroke_width=1.5, dash_length=0.12
        )
        h_proj = DashedLine(
            axes.c2p(0, fy_val), axes.c2p(fx_val, fy_val),
            color=S.orange, stroke_width=1.5, dash_length=0.12
        )
        x_tick_label = MathTex(r"x=1.5", color=S.orange, font_size=26
                                ).next_to(axes.c2p(fx_val, 0), DOWN, buff=0.18)
        y_tick_label = MathTex(r"y=2.25", color=S.orange, font_size=26
                                ).next_to(axes.c2p(0, fy_val), LEFT, buff=0.18)

        point_card = self._info_card([
            r"\textbf{Point:} \ (1.5,\ 2.25)",
            r"\text{Slope of the CURVE here?}"
        ], width=7.2)

        self.play(
            Create(v_proj), Create(h_proj),
            FadeIn(focus_dot, scale=0.4),
            run_time=1.0
        )
        self.play(
            Write(x_tick_label), Write(y_tick_label),
            FadeIn(point_card, shift=0.2*UP),
            run_time=0.9
        )
        self.wait(1.5)
        self.play(FadeOut(point_card), FadeOut(x_tick_label),
                  FadeOut(y_tick_label), FadeOut(v_proj), FadeOut(h_proj),
                  run_time=0.4)

        # ── SCENE 3: secant introduction ────────────────
        h_tracker = ValueTracker(P.h_start)

        nearby_dot = always_redraw(lambda: Dot(
            axes.c2p(P.focus_x + h_tracker.get_value(),
                     (P.focus_x + h_tracker.get_value()) ** 2),
            radius=0.08, color=S.cyan, z_index=3
        ))

        secant_line = always_redraw(lambda: self._secant(axes, h_tracker.get_value()))

        dx_brace = always_redraw(lambda: BraceBetweenPoints(
            axes.c2p(P.focus_x, -0.28),
            axes.c2p(P.focus_x + h_tracker.get_value(), -0.28),
            direction=DOWN, color=S.cyan
        ).set_stroke(width=1))
        h_label = always_redraw(lambda: MathTex(
            r"h", color=S.cyan, font_size=26
        ).next_to(
            axes.c2p((P.focus_x + P.focus_x + h_tracker.get_value()) / 2, -0.28),
            DOWN, buff=0.22
        ))

        dx_line = always_redraw(lambda: DashedLine(
            axes.c2p(P.focus_x, P.focus_x ** 2),
            axes.c2p(P.focus_x + h_tracker.get_value(), P.focus_x ** 2),
            color=S.cyan, stroke_width=2, dash_length=0.1
        ))
        dx_label = always_redraw(lambda: MathTex(
            r"\Delta x", color=S.cyan, font_size=24
        ).next_to(
            axes.c2p((P.focus_x + P.focus_x + h_tracker.get_value()) / 2, P.focus_x ** 2),
            UP, buff=0.12
        ))

        dy_line = always_redraw(lambda: DashedLine(
            axes.c2p(P.focus_x + h_tracker.get_value(),
                     P.focus_x ** 2),
            axes.c2p(P.focus_x + h_tracker.get_value(),
                     (P.focus_x + h_tracker.get_value()) ** 2),
            color=S.magenta, stroke_width=2, dash_length=0.1
        ))
        dy_label = always_redraw(lambda: MathTex(
            r"\Delta y", color=S.magenta, font_size=24
        ).next_to(
            axes.c2p(P.focus_x + h_tracker.get_value(),
                     (P.focus_x ** 2 + (P.focus_x + h_tracker.get_value()) ** 2) / 2),
            RIGHT, buff=0.12
        ))

        slope_readout = always_redraw(lambda: self._slope_readout(h_tracker.get_value()))

        secant_card = self._info_card([
            r"\text{Secant line connects two points}",
            r"\text{Slope} = \frac{\Delta y}{\Delta x} = \frac{f(x+h)-f(x)}{h}",
        ], width=8.0)

        self.play(
            FadeIn(nearby_dot), Create(secant_line),
            run_time=0.9
        )
        self.play(
            Create(dx_brace), Write(h_label), Create(dx_line), Write(dx_label),
            Create(dy_line), Write(dy_label),
            FadeIn(secant_card, shift=0.2*UP),
            FadeIn(slope_readout),
            run_time=1.1
        )
        self.wait(2.0)
        self.play(FadeOut(secant_card), run_time=0.4)

        # ── SCENE 4: squeeze h → 0 ────────────────
        squeeze_card = self._info_card([
            r"\text{Watch the slope as } h \to 0"
        ], width=6.5)
        self.play(FadeIn(squeeze_card, shift=0.2*UP), run_time=0.5)
        self.wait(0.4)

        self.play(h_tracker.animate.set_value(0.6), run_time=2.0, rate_func=smooth)
        self.wait(0.4)
        self.play(h_tracker.animate.set_value(0.2), run_time=2.5, rate_func=smooth)
        self.wait(0.4)
        self.play(h_tracker.animate.set_value(P.h_end), run_time=3.5, rate_func=smooth)
        self.wait(0.6)
        self.play(FadeOut(squeeze_card), run_time=0.3)

        # ── SCENE 5: zoom-to-linear insight ──────────────
        zoom_card = self._info_card([
            r"\text{Zoom in: the curve } \textit{becomes} \text{ a line}"
        ], width=7.5)
        self.play(FadeIn(zoom_card, shift=0.2*UP), run_time=0.5)
        self.wait(0.5)

        fp = axes.c2p(P.focus_x, P.focus_x**2)
        self.play(
            self.camera.frame.animate
                .set(width=1.8)
                .move_to(fp),
            run_time=3.0, rate_func=smooth
        )
        self.wait(2.0)

        self.play(
            self.camera.frame.animate
                .set(width=config.frame_width)
                .move_to(ORIGIN),
            run_time=2.0, rate_func=smooth
        )
        self.play(FadeOut(zoom_card), run_time=0.3)

        # ── SCENE 6: reveal the tangent & derivative ────────────────
        self.play(
            FadeOut(nearby_dot), FadeOut(dx_brace), FadeOut(dx_line), FadeOut(dx_label),
            FadeOut(h_label), FadeOut(dy_line), FadeOut(dy_label),
            FadeOut(slope_readout),
            run_time=0.6
        )

        tangent_line = self._tangent(axes, P.focus_x)
        self.play(Transform(secant_line, tangent_line), run_time=1.0)

        lim_formula = MathTex(
            r"f'(x) = \lim_{h \to 0} \frac{f(x+h)-f(x)}{h}",
            color=S.text, font_size=34
        ).to_edge(UP, buff=0.55)

        specific_formula = MathTex(
            r"f'(x) = 2x",
            color=S.orange, font_size=40
        ).next_to(lim_formula, DOWN, buff=0.4)

        at_point = MathTex(
            r"f'(1.5) = 2 \times 1.5 = 3",
            color=S.green, font_size=34
        ).next_to(specific_formula, DOWN, buff=0.35)

        self.play(Write(lim_formula), run_time=1.5)
        self.wait(0.8)
        self.play(Write(specific_formula), run_time=1.2)
        self.wait(0.8)
        self.play(Write(at_point), run_time=1.0)
        self.wait(1.5)

        slope_label = MathTex(r"\text{slope} = 3", color=S.orange, font_size=28
                               ).next_to(secant_line, UR, buff=0.18)
        self.play(FadeIn(slope_label, shift=0.1*UP), run_time=0.6)
        self.wait(1.0)

        self.play(
            FadeOut(lim_formula), FadeOut(specific_formula),
            FadeOut(at_point), FadeOut(slope_label),
            run_time=0.5
        )

        # ── SCENE 7: generalise — tangent travels the curve ───────────────
        x_tracker = ValueTracker(P.focus_x)

        roving_dot = always_redraw(lambda: Dot(
            axes.c2p(x_tracker.get_value(), x_tracker.get_value()**2),
            radius=0.10, color=S.orange, z_index=3
        ))
        roving_tangent = always_redraw(lambda: self._tangent(
            axes, x_tracker.get_value()
        ))
        slope_annotation = always_redraw(lambda: MathTex(
            r"f'(x) = " + f"{2*x_tracker.get_value():.2f}",
            color=S.orange, font_size=32
        ).to_edge(UP, buff=0.55))

        generalise_card = self._info_card([
            r"f'(x) = 2x \quad \text{for any } x",
            r"\text{The slope changes along the curve}"
        ], width=7.8)

        self.play(
            FadeOut(secant_line), FadeOut(focus_dot),
            FadeIn(roving_dot), FadeIn(roving_tangent),
            FadeIn(slope_annotation),
            FadeIn(generalise_card, shift=0.2*UP),
            run_time=0.8
        )

        self.play(x_tracker.animate.set_value(0.05),  run_time=2.5, rate_func=smooth)
        self.wait(0.3)
        self.play(x_tracker.animate.set_value(2.4),   run_time=4.5, rate_func=smooth)
        self.wait(0.4)
        self.play(x_tracker.animate.set_value(P.focus_x), run_time=1.5, rate_func=smooth)
        self.wait(0.8)

        self.play(
            FadeOut(generalise_card), FadeOut(roving_tangent),
            FadeOut(roving_dot), FadeOut(slope_annotation),
            run_time=0.5
        )

        # ── SCENE 8: closing title card ───────────────
        closing = self._closing_card()
        self.play(FadeIn(closing), run_time=0.5)
        self.wait(1.5)
        self.play(FadeOut(closing), run_time=0.5)

    # ─────────────────────────────────
    #  Factory helpers
    # ─────────────────────────────────

    def _make_axes(self) -> Axes:
        ax = Axes(
            x_range=[P.x_min, P.x_max, 1],
            y_range=[P.y_min, P.y_max, 1],
            x_length=6.8,
            y_length=9.2,
            tips=True,
            axis_config={                "color": WHITE,
                "stroke_width": 1.5,
                "include_numbers": True,
            },
        ).shift(DOWN * 0.4)
        for mob in ax.get_axis_labels():
            mob.set_color(S.muted)
        return ax

    def _secant(self, axes: Axes, h: float) -> Line:
        x0, x1  = P.focus_x, P.focus_x + h
        y0, y1  = x0**2, x1**2
        p0 = np.array(axes.c2p(x0, y0))
        p1 = np.array(axes.c2p(x1, y1))
        direction = (p1 - p0) / np.linalg.norm(p1 - p0)
        ext = 1.5
        return Line(
            p0 - ext * direction,
            p1 + ext * direction,
            color=S.magenta, stroke_width=3.5
        )

    def _tangent(self, axes: Axes, x: float, ext: float = 1.7) -> Line:
        """True tangent at data point x, properly scaled for screen space."""
        slope = _screen_slope(axes, x, 2 * x)
        p     = np.array(axes.c2p(x, x**2))
        angle = np.arctan(slope)
        direction = np.array([np.cos(angle), np.sin(angle), 0])
        return Line(
            p - ext * direction,
            p + ext * direction,
            color=S.orange, stroke_width=4.5
        )

    def _slope_readout(self, h: float) -> VGroup:
        slope_val = (((P.focus_x + h)**2) - P.focus_x**2) / h
        label = VGroup(
            Text("Secant slope:", color=S.muted, font_size=22),
            Text(f"{slope_val:.3f}", color=S.magenta,
                 font_size=36, weight=BOLD),
            Text(f"(h = Δx = {h:.3f})", color=S.muted, font_size=20),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.12)

        bg = RoundedRectangle(
            width=label.width + 0.7, height=label.height + 0.4,
            corner_radius=0.15,
            fill_color=S.card_fill, fill_opacity=0.92,
            stroke_color=S.card_stroke, stroke_width=1
        ).move_to(label)
        return VGroup(bg, label).to_corner(UR, buff=0.35).shift(DOWN * 0.2)

    def _info_card(self, tex_lines: list[str], width: float = 7.0) -> VGroup:
        """Bottom information card with LaTeX lines."""
        lines = VGroup(*[
            MathTex(t, color=S.text, font_size=30) for t in tex_lines
        ]).arrange(DOWN, aligned_edge=LEFT, buff=0.28)

        bg = RoundedRectangle(
            width=width, height=lines.height + 0.65,
            corner_radius=0.2,
            fill_color=S.card_fill, fill_opacity=0.93,
            stroke_color=S.card_stroke, stroke_width=1.2
        ).move_to(lines)

        card = VGroup(bg, lines)
        card.to_edge(DOWN, buff=0.45)
        return card

    def _title(self, txt: str) -> Text:
        return Text(txt, font_size=40, color=S.text,
                    weight=BOLD, line_spacing=1.1
                    ).to_edge(UP, buff=0.5)

    def _watermark(self) -> Text:
        return Text(
            ") Mugambi Ndwiga / @craftsandengineering",
            font_size=17, color=S.muted,
            slant=ITALIC
        ).set_opacity(0.6)

    def _closing_card(self) -> VGroup:
        bg = Rectangle(
            width=config.frame_width + 0.2,
            height=config.frame_height + 0.2,
            fill_color=S.bg, fill_opacity=1,
            stroke_width=0
        )
        line1 = Text("Made by Mugambi Ndwiga",
                     font_size=36, color=S.text, weight=BOLD)
        line2 = Text("@craftsandengineering",
                     font_size=28, color=S.cyan)
        lines = VGroup(line1, line2).arrange(DOWN, buff=0.35)
        return VGroup(bg, lines)

Manim Community v0.18.1

In [3]:
import glob, os, subprocess, json
from IPython.display import Video, display

# Find the most recently rendered video file
matches = glob.glob("media/videos/**/derivative_tangent.mp4", recursive=True)
if not matches:
    print("No video found. Please run the Manim cell above first.")
else:
    path = max(matches, key=os.path.getmtime)

    # Probe video metadata
    probe = json.loads(subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=width,height", "-of", "json", path],
        capture_output=True, text=True).stdout)

    w = probe["streams"][0]["width"]
    h = probe["streams"][0]["height"]
    size_mb = os.path.getsize(path) / (1024 * 1024)

    print(f"--- Video Metadata ---")
    print(f"File: {path}")
    print(f"Resolution: {w}x{h}")
    print(f"Size: {size_mb:.2f} MB")
    print("---------------------")

    # Displaying with scaled dimensions for the notebook while maintaining 9:16
    # width=320 is a good size for side-by-side viewing in Colab
    display(Video(path, embed=True, width=320, html_attributes='controls autoplay loop style="max-width: 100%; border-radius: 8px; shadow: 0 4px 8px rgba(0,0,0,0.2);"'))

--- Video Metadata ---
File: media/videos/content/1920p30/derivative_tangent.mp4
Resolution: 1080x1920
Size: 2.49 MB
---------------------
